In [1]:
# Make `src` importable from notebooks/ and auto-reload edits to .py files
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
from src.backbone.common import get_activation, FrozenBatchNorm2d, ConvNormLayer

torch.manual_seed(0)
print("torch", torch.__version__)

torch 2.13.0+cu126


In [2]:
# None → a real Identity layer (not the class, not None)
a = get_activation(None)
assert isinstance(a, nn.Identity), type(a)

# Case-insensitive strings → fresh instances
for name, cls in [("relu", nn.ReLU), ("SiLU", nn.SiLU), ("GELU", nn.GELU), ("leaky_relu", nn.LeakyReLU)]:
    layer = get_activation(name)
    assert isinstance(layer, cls), f"{name}: got {type(layer)}"

# Each call gives a NEW object
assert get_activation("relu") is not get_activation("relu")

# A Module passes through unchanged (same object)
g = nn.GELU()
assert get_activation(g) is g

# Unknown name → ValueError that shows the original spelling
try:
    get_activation("FooBar")
    raise AssertionError("should have raised")
except ValueError as e:
    assert "FooBar" in str(e), str(e)
    print("ValueError OK:", e)

# Wrong type → TypeError (e.g. passing the class instead of an instance)
try:
    get_activation(nn.ReLU)
    raise AssertionError("should have raised")
except TypeError as e:
    print("TypeError OK:", e)

print("✅ get_activation")

ValueError OK: Unknown activation 'FooBar'. Valid options: ['gelu', 'leaky_relu', 'relu', 'silu']
TypeError OK: activation must be str, nn.Module or None, got type
✅ get_activation


In [3]:
C = 8
bn = nn.BatchNorm2d(C)
with torch.no_grad():
    bn.weight.copy_(torch.randn(C))
    bn.bias.copy_(torch.randn(C))
    bn.running_mean.copy_(torch.randn(C))
    bn.running_var.copy_(torch.rand(C) + 0.1)  # keep variance positive
bn.eval()

frozen = FrozenBatchNorm2d(C)
# strict=True on purpose: crashes if buffer names don't match BN's keys
result = frozen.load_state_dict(bn.state_dict(), strict=True)
print("load result:", result)

x = torch.randn(2, C, 4, 4)
max_diff = (bn(x) - frozen(x)).abs().max().item()
print(f"max |bn - frozen| = {max_diff:.2e}")
assert torch.allclose(bn(x), frozen(x), atol=1e-6)

print("✅ FrozenBatchNorm2d matches BatchNorm2d.eval()")

load result: <All keys matched successfully>
max |bn - frozen| = 4.77e-07
✅ FrozenBatchNorm2d matches BatchNorm2d.eval()


In [4]:
# No trainable parameters at all
assert len(list(frozen.parameters())) == 0, list(frozen.named_parameters())

# But the 4 buffers ARE saved (and no num_batches_tracked)
keys = sorted(frozen.state_dict().keys())
print("state_dict keys:", keys)
assert keys == ["bias", "running_mean", "running_var", "weight"]

# Output does not change between train() and eval() (no stat updates)
frozen.train()
y_train = frozen(x)
frozen.eval()
y_eval = frozen(x)
assert torch.equal(y_train, y_eval)

# Gradients still flow THROUGH it to the input
x_req = torch.randn(2, C, 4, 4, requires_grad=True)
frozen(x_req).sum().backward()
assert x_req.grad is not None

# repr looks right (not doubled)
print(frozen)
assert repr(frozen) == "FrozenBatchNorm2d(8, eps=1e-05)"

print("✅ FrozenBatchNorm2d is frozen, saves buffers, passes gradients")

state_dict keys: ['bias', 'running_mean', 'running_var', 'weight']
FrozenBatchNorm2d(8, eps=1e-05)
✅ FrozenBatchNorm2d is frozen, saves buffers, passes gradients


In [5]:
# (ch_in, ch_out, k, s, input HxW, expected HxW)
# Use ch_in != ch_out on purpose so a wrong BN size would crash
cases = [
    (3, 16, 3, 1, (64, 64), (64, 64)),   # 3x3 s1 keeps size
    (3, 16, 3, 2, (64, 64), (32, 32)),   # 3x3 s2 halves
    (3, 16, 1, 1, (64, 64), (64, 64)),   # 1x1 keeps size
    (16, 32, 3, 2, (65, 65), (33, 33)),  # odd size: ceil(65/2) = 33
    (16, 32, 3, 2, (40, 56), (20, 28)),  # non-square
]

for ch_in, ch_out, k, s, (h, w), (eh, ew) in cases:
    layer = ConvNormLayer(ch_in, ch_out, k, s, act="silu")
    out = layer(torch.randn(2, ch_in, h, w))
    expected = (2, ch_out, eh, ew)
    status = "OK " if tuple(out.shape) == expected else "BAD"
    print(f"{status} k={k} s={s} {ch_in}->{ch_out} {h}x{w} → {tuple(out.shape)} (want {expected})")
    assert tuple(out.shape) == expected

print("✅ ConvNormLayer shapes")

OK  k=3 s=1 3->16 64x64 → (2, 16, 64, 64) (want (2, 16, 64, 64))
OK  k=3 s=2 3->16 64x64 → (2, 16, 32, 32) (want (2, 16, 32, 32))
OK  k=1 s=1 3->16 64x64 → (2, 16, 64, 64) (want (2, 16, 64, 64))
OK  k=3 s=2 16->32 65x65 → (2, 32, 33, 33) (want (2, 32, 33, 33))
OK  k=3 s=2 16->32 40x56 → (2, 32, 20, 28) (want (2, 32, 20, 28))
✅ ConvNormLayer shapes


In [6]:
layer = ConvNormLayer(3, 16, 3, 2)   # act=None
print(layer)

# Sub-module names must be conv / norm / act (needed for weight loading later)
names = [n for n, _ in layer.named_children()]
assert names == ["conv", "norm", "act"], names

# Conv has no bias (BN's bias does that job)
assert layer.conv.bias is None

# Norm size matches OUTPUT channels
assert layer.norm.num_features == 16

# act=None → Identity
assert isinstance(layer.act, nn.Identity)

# Explicit padding override still works
layer_p0 = ConvNormLayer(3, 16, 3, 1, padding=0)
assert layer_p0(torch.randn(1, 3, 10, 10)).shape[-1] == 8   # 10 - 3 + 1

# Parameter count: conv 3*16*3*3 = 432, BN weight+bias = 32 → 464
n_params = sum(p.numel() for p in layer.parameters())
print("params:", n_params)
assert n_params == 3 * 16 * 3 * 3 + 2 * 16

print("✅ ConvNormLayer structure")

ConvNormLayer(
  (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (norm): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act): Identity()
)
params: 464
✅ ConvNormLayer structure


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
from src.backbone.common import ConvNormLayer
from src.backbone.presnet import BasicBlock, BottleNeck, Blocks

torch.manual_seed(0)

In [3]:
# (block, ch_in, width, count, stride, input HxW, expected out_ch, expected HxW)
cases = [
    (BasicBlock, 64, 64, 2, 1, (64, 64), 64, (64, 64)),     # R18 stage 1
    (BasicBlock, 64, 128, 2, 2, (64, 64), 128, (32, 32)),   # R18 stage 2
    (BottleNeck, 64, 64, 3, 1, (64, 64), 256, (64, 64)),    # R50 stage 1
    (BottleNeck, 256, 128, 4, 2, (64, 64), 512, (32, 32)),  # R50 stage 2
    (BottleNeck, 256, 128, 4, 2, (65, 65), 512, (33, 33)),  # odd size
]
for blk, ch_in, w, n, s, (h, wd), oc, (eh, ew) in cases:
    stage = Blocks(blk, ch_in, w, count=n, stride=s)
    out = stage(torch.randn(2, ch_in, h, wd))
    want = (2, oc, eh, ew)
    print(f"{'OK ' if tuple(out.shape) == want else 'BAD'} {blk.__name__:10s} x{n} s={s} → {tuple(out.shape)}")
    assert tuple(out.shape) == want
    assert stage.out_channels == oc
print("✅ Blocks shapes + out_channels")

OK  BasicBlock x2 s=1 → (2, 64, 64, 64)
OK  BasicBlock x2 s=2 → (2, 128, 32, 32)
OK  BottleNeck x3 s=1 → (2, 256, 64, 64)
OK  BottleNeck x4 s=2 → (2, 512, 32, 32)
OK  BottleNeck x4 s=2 → (2, 512, 33, 33)
✅ Blocks shapes + out_channels


In [4]:
stage = Blocks(BottleNeck, 256, 128, count=4, stride=2)
blocks = list(stage.blocks)
assert len(blocks) == 4

# Only block 0 projects
assert hasattr(blocks[0], "short") and not blocks[0].shortcut
for i, b in enumerate(blocks[1:], start=1):
    assert not hasattr(b, "short") and b.shortcut, f"block {i} should be identity"

# Only block 0 downsamples (stride lives on the 3x3 in BottleNeck)
assert blocks[0].branch2b.conv.stride == (2, 2)
assert all(b.branch2b.conv.stride == (1, 1) for b in blocks[1:])

# ch_in chaining: block 1 must take 128*4 = 512 channels, not 256
assert blocks[1].branch2a.conv.weight.shape[1] == 512, "ch_in not updated after block 0"

# Official-style key names
keys = stage.state_dict().keys()
for k in ["blocks.0.short.conv.conv.weight", "blocks.0.branch2c.norm.running_var",
          "blocks.3.branch2a.conv.weight"]:
    assert k in keys, f"missing {k}"
assert not any(k.startswith("blocks.1.short") for k in keys)
print("✅ Blocks structure")

✅ Blocks structure


In [5]:
# count=1 → just the projection block
stage = Blocks(BasicBlock, 64, 128, count=1, stride=2)
assert len(stage.blocks) == 1
assert stage(torch.randn(1, 64, 16, 16)).shape == (1, 128, 8, 8)

# count=0 → fail early with ValueError (not a later AttributeError)
try:
    Blocks(BasicBlock, 64, 64, count=0, stride=1)
    raise AssertionError("should have raised")
except ValueError as e:
    print("ValueError OK:", e)
print("✅ count edge cases")

ValueError OK: Counts should be >= 1, but got 0
✅ count edge cases


In [6]:
stage = Blocks(BottleNeck, 64, 64, count=3, stride=1)
x = torch.randn(2, 64, 16, 16, requires_grad=True)
stage(x).sum().backward()
assert x.grad is not None
assert all(p.grad is not None for p in stage.parameters()), "some params got no grad"
print("✅ gradients flow")

!cd {ROOT} && ruff check src/backbone/presnet.py

✅ gradients flow
All checks passed!
